# Train on Existing Kaggle Data (No Upload Needed!)

**Uses pre-uploaded datasets:**
- subhajitdas/chuckle-wavlm-training-data (87 videos, F1=0.96)
- subhajitdas/gillick272-prosody (272 videos, real labels)
- subhajitdas/top200-youtube-comedy-prosody (200 videos, 15-dim features)

**Goal:** Train best model and download for local use

In [ ]:
# === SETUP ===
!pip install -q kaggle
!pip install -q numpy pandas scikit-learn

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
import pickle
import os

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete')

In [ ]:
# === LOAD KAGGLE DATASETS ===
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

print('=== Loading 87-video dataset ===')
api.dataset_download_files('subhajitdas/chuckle-wavlm-training-data', 
                          path=OUTPUT_DIR, unzip=True)

# Find the npz file
npz_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.npz')]
print(f'Found files: {npz_files}')

In [ ]:
# === TRAIN ON 87-VIDEO DATA ===
npz_path = f'{OUTPUT_DIR}/{npz_files[0]}'
data = np.load(npz_path)

X = data['features'] if 'features' in data else data['X']
y = data['labels'] if 'labels' in data else data['y']

print(f'Features shape: {X.shape}')
print(f'Labels: pos={y.sum()}/{len(y)} ({100*y.mean():.1f}%)')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)}, pos={y_train.sum()}')
print(f'Test: {len(X_test)}, pos={y_test.sum()}')

In [ ]:
# === TRAIN MODEL ===
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Reacall: {rec:.4f}')

In [ ]:
# === SAVE MODEL ===
model_data = {
    'model': clf,
    'scaler': scaler,
    'f1': f1,
    'precision': prec,
    'recall': rec,
    'n_train': len(X_train),
    'n_test': len(X_test)
}

with open(f'{OUTPUT_DIR}/laughter_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f'Model saved to {OUTPUT_DIR}/laughter_model.pkl')
print('\nDownload from Kaggle output!')